In [ ]:
# pip install transformers torch opencv-python pillow numpy pandas

import re
import time
from multiprocessing import Pool
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

LABELS_CSV = Path("outputs/volume_surface_error.csv")
BGR_DIR = Path("outputs/bgr_images")
CACHE_DIR = Path("outputs/cache")

CACHE_SIZE = 384
WHITE_THRESHOLD = 250
MARGIN = 0.08
TEST_PARTS = {"Plug"}
DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Large-hf"
BATCH = 32
WORKERS = 32
VIEW_RE = re.compile(r"azimuth_(?P<az>\d+)_polar_(?P<pol>\d+)$")


def crop_pad_resize(path):
    with Image.open(path) as im:
        a = np.asarray(im.convert("RGB"))
    ys, xs = np.nonzero((a < WHITE_THRESHOLD).any(axis=2))
    if len(ys) == 0:
        y0, y1, x0, x1 = 0, a.shape[0], 0, a.shape[1]
    else:
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    h, w = y1 - y0, x1 - x0
    side = int(max(h, w) * (1 + 2 * MARGIN))
    canvas = np.full((side, side, 3), 255, dtype=np.uint8)
    oy, ox = (side - h) // 2, (side - w) // 2
    canvas[oy:oy + h, ox:ox + w] = a[y0:y1, x0:x1]
    return np.asarray(Image.fromarray(canvas).resize((CACHE_SIZE, CACHE_SIZE), Image.Resampling.LANCZOS), dtype=np.uint8)


df = pd.read_csv(LABELS_CSV).dropna(subset=["surface_dft_um_mean"])
parts = df["folder_path"].str.split("_", expand=True)
df["part"], df["texture"], df["replica"] = parts[0], parts[1], parts[2]
views = df["image_name"].str.extract(VIEW_RE).astype(int)
df["azimuth"], df["polar"] = views["az"], views["pol"]
df["group"] = df["folder_path"]
df["split"] = np.where(df["part"].isin(TEST_PARTS), "test", "train")
df["path"] = [str(BGR_DIR / p / t / r / f"{n}.png") for p, t, r, n in zip(df["part"], df["texture"], df["replica"], df["image_name"])]
df = df.sort_values(["split", "part", "texture", "replica", "azimuth", "polar"],
                    ascending=[False, True, True, True, True, True]).reset_index(drop=True)

images = np.empty((len(df), CACHE_SIZE, CACHE_SIZE, 3), dtype=np.uint8)
with Pool(WORKERS) as pool:
    for i, arr in enumerate(pool.imap(crop_pad_resize, df["path"].tolist(), chunksize=4)):
        images[i] = arr
CACHE_DIR.mkdir(parents=True, exist_ok=True)
np.save(CACHE_DIR / f"images_{CACHE_SIZE}.npy", images)
df.drop(columns=["path"]).to_csv(CACHE_DIR / "meta.csv", index=False)
print(f"{len(df)} images cached", flush=True)

proc = AutoImageProcessor.from_pretrained(DEPTH_MODEL)
model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL).to("cuda").eval()
depth_maps = np.zeros((len(df), CACHE_SIZE, CACHE_SIZE), dtype=np.float16)
t0 = time.time()
for s in range(0, len(df), BATCH):
    e = min(s + BATCH, len(df))
    inputs = proc(images=[np.ascontiguousarray(images[i]) for i in range(s, e)], return_tensors="pt").to("cuda")
    with torch.no_grad():
        depth = model(**inputs).predicted_depth.float().cpu().numpy()
    for bi, i in enumerate(range(s, e)):
        d = cv2.resize(depth[bi], (CACHE_SIZE, CACHE_SIZE), interpolation=cv2.INTER_LINEAR)
        depth_maps[i] = ((d - d.min()) / (d.max() - d.min() + 1e-9)).astype(np.float16)
    if (e // BATCH) % 20 == 0 or e == len(df):
        print(f"depth {e}/{len(df)}  {time.time() - t0:.0f}s", flush=True)
np.save(CACHE_DIR / "depth_maps.npy", depth_maps)